# Count and Binary Spatial Model Estimation

This notebook demonstrates **cross-sectional spatial model estimation** in `bayespecon` for two outcome types:

1. **Negative Binomial (NegBin)** — overdispersed count outcomes, using either NUTS (via PyMC) or a Pólya–Gamma Gibbs sampler
2. **Binary (Logit)** — binary outcomes with spatial dependence, using a Pólya–Gamma Gibbs sampler

Both model families use structural-form parameterisations and produce `arviz.InferenceData` output compatible with all downstream diagnostics.

## When to Use NUTS vs Pólya–Gamma Gibbs

| Criterion | NUTS (`SARNegativeBinomial`) | PG-Gibbs (`SARNegBinLatent`) |
|---|---|---|
| **Model form** | Non-centred: $\eta = (I - \rho W)^{-1}(X\beta + \sigma z)$ | Structural: $\eta = \rho W \eta + X\beta + \nu$ |
| **Spatial Jacobian** | Not needed (cancels in non-centred form) | Not needed (PG augmentation avoids it) |
| **Speed** | Slow for large $n$ (gradient through sparse solve) | Fast (conjugate blocks for $\beta$, $\sigma^2$, $\eta$; slice for $\rho$) |
| **ESS/s** | Low for spatial models | High (conjugate blocks mix well) |
| **Tuning** | 1000+ tuning steps needed | Minimal (adaptive slice width or MALA step size) |
| **PyMC model graph** | ✅ Full access | ❌ Bypasses PyMC entirely |
| **Custom inference** | ✅ `pm.sample_prior_predictive`, etc. | ❌ Not available |
| **Warm start** | ❌ PyMC default init | ✅ GLM-based initialization |

**Rule of thumb**: Use `SARNegBinLatent` for $n > 500$ where NUTS ESS/s is poor. Use `SARNegativeBinomial` for small $n$ or when you need the full PyMC model graph.

In [ ]:
%load_ext autoreload
%autoreload 2

import arviz as az
import numpy as np

from bayespecon.dgp import simulate_sar_negbin
from bayespecon.models import SARNegativeBinomial, SARNegBinLatent

## Generate Data with Native DGP Function

We use `simulate_sar_negbin` from `bayespecon.dgp` to generate synthetic count data from a known SAR-NB2 process. This function supports two regimes:

- **`sigma2 = 0`** (default): Deterministic reduced form $\eta = (I - \rho W)^{-1} X\beta$, matching `SARNegativeBinomial`.
- **`sigma2 > 0`**: Structural form with Gaussian noise $\eta = \rho W \eta + X\beta + \nu$, matching `SARNegBinLatent`.

We use `sigma2 > 0` here so both model classes can recover the true parameters.

In [ ]:
# Generate SAR-NB2 data on a 10x10 rook grid (n=100)
data = simulate_sar_negbin(
    n=10,  # 10x10 grid → 100 observations
    rho=0.5,
    beta=np.array([1.0, 0.6, -0.3]),  # intercept + 2 covariates
    alpha=2.0,  # NB2 dispersion
    sigma2=0.5,  # structural-form residual variance
    seed=42,
)

y = data["y"]
X = data["X"]
W = data["W_graph"]

print(f"n = {len(y)}, k = {X.shape[1]}")
print(
    f"y range: [{y.min():.0f}, {y.max():.0f}], mean = {y.mean():.1f}, var = {y.var():.1f}"
)
print(f"True params: {data['params_true']}")

In [ ]:
# Generate SAR-NB2 data on a 10x10 rook grid (n=100)
gdf = simulate_sar_negbin(
    n=10,  # 10x10 grid → 100 observations
    rho=0.5,
    beta=np.array([1.0, 0.6, -0.3]),  # intercept + 2 covariates
    alpha=2.0,  # NB2 dispersion
    sigma2=0.5,  # structural-form residual variance
    seed=42,
    create_gdf=True,
)
gdf.plot("y")

In [ ]:
from esda import G

In [ ]:
gstat = G(y, W)
gstat.p_sim

## NUTS Estimation: `SARNegativeBinomial`

The `SARNegativeBinomial` model uses PyMC's NUTS sampler with a **non-centred parameterisation**:

$$\eta = (I - \rho W)^{-1}(X\beta + \sigma z), \quad z \sim N(0, I)$$

No spatial Jacobian $\log|I - \rho W|$ is needed because the change-of-variables Jacobian cancels with the multivariate-normal normalisation constant. This is the key advantage of the non-centred form for NUTS.

In [ ]:
model_nuts = SARNegativeBinomial(y=y, X=X, W=W)
idata_nuts = model_nuts.fit(
    draws=2000,
    tune=1000,
    chains=4,
    target_accept=0.9,
    random_seed=42,
    idata_kwargs={"log_likelihood": True},
)

print("NUTS posterior means:")
print(f"  rho   = {float(idata_nuts.posterior['rho'].mean()):.4f}  (true = 0.5)")
print(
    f"  beta  = {idata_nuts.posterior['beta'].mean(dim=['chain', 'draw']).values}  (true = [1.0, 0.6, -0.3])"
)
print(
    f"  sigma = {float(idata_nuts.posterior['sigma'].mean()):.4f}  (true = {np.sqrt(0.5):.4f})"
)
print(f"  alpha = {float(idata_nuts.posterior['alpha'].mean()):.4f}  (true = 2.0)")

In [ ]:
az.summary(idata_nuts, var_names=["rho", "beta", "sigma", "alpha"])

## Pólya–Gamma Gibbs: `SARNegBinLatent`

The `SARNegBinLatent` model uses a **structural-form** parameterisation with Pólya–Gamma data augmentation:

$$y_i \sim \mathrm{NegBin}(\exp(\eta_i), \alpha), \quad \eta = \rho W \eta + X\beta + \nu, \quad \nu \sim N(0, \sigma^2 I)$$

The Gibbs sampler uses a **5-block** strategy:
1. **ω | η, y, α** — Pólya–Gamma auxiliary variables (conjugate)
2. **η | ω, β, σ², ρ** — Multivariate normal (conjugate, sparse precision solve)
3. **β | η, ω, σ²** — Multivariate normal (conjugate)
4. **σ² | η, β, ω** — Inverse-gamma (conjugate)
5. **ρ | η, β, σ²** — 1-D slice sampling or MALA (non-conjugate, scalar)
6. **α | η, y** — Griddy-Gibbs or slice (non-conjugate, scalar)

Only ρ and α are non-conjugate, and both are scalars.

In [ ]:
model_gibbs = SARNegBinLatent(y=y, X=X, W=W)
idata_gibbs = model_gibbs.fit(
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
    n_jobs=-1,
)

print("PG-Gibbs posterior means:")
print(f"  rho   = {float(idata_gibbs.posterior['rho'].mean()):.4f}  (true = 0.5)")
print(
    f"  beta  = {idata_gibbs.posterior['beta'].mean(dim=['chain', 'draw']).values}  (true = [1.0, 0.6, -0.3])"
)
print(
    f"  sigma = {float(idata_gibbs.posterior['sigma'].mean()):.4f}  (true = {np.sqrt(0.5):.4f})"
)
print(f"  alpha = {float(idata_gibbs.posterior['alpha'].mean()):.4f}  (true = 2.0)")

In [ ]:
az.summary(idata_gibbs, var_names=["rho", "beta", "sigma", "alpha"])

In [ ]:
az.plot_trace(idata_gibbs)

## Gibbs Backend Options

The Pólya–Gamma Gibbs sampler supports multiple execution backends:

| Backend | `gibbs_method` | ρ update | α update | η solve | Requires |
|---|---|---|---|---|---|
| **CHOLMOD** | `"auto"` (default when available) | Adaptive slice | Griddy-Gibbs | CHOLMOD factorisation | scikit-sparse |
| **SPLU** | `"auto"` (fallback) | Adaptive slice | Griddy-Gibbs | `scipy.sparse.linalg.splu` | SciPy only |
| **JAX dense** | `"jax_dense"` | Slice (Neal stepping-out) | Slice (JAX-compiled) | JAX dense Cholesky | JAX |

### Auto-selection logic

When `gibbs_method="auto"` (the default):
1. If CHOLMOD is available → use `"factorize"` (fastest on CPU for sparse W)
2. If JAX is installed and $n \leq 10\,000$ → use `"jax_dense"`
3. Otherwise → fall back to SPLU

### Warm start

`SARNegBinLatent` initialises from a non-spatial NB GLM fit via method of moments, providing reasonable starting values for all parameters. This avoids the long burn-in that NUTS requires.

## JAX Backend for Negative Binomial

The JAX path compiles the entire Pólya–Gamma Gibbs step into a single XLA kernel, eliminating Python dispatch overhead. Both ρ and α use **JAX-compiled slice sampling** (Neal 2003 stepping-out), which provides better mixing than MALA or random-walk Metropolis–Hastings for spatial autoregressive parameters.

### Architecture

| Block | Parameter | Sampler | Notes |
|---|---|---|---|
| 1 | ω | Pólya–Gamma (sum-of-exponentials) | JIT-compiled, K=10 terms |
| 2 | η | Dense Cholesky solve | O(n³), fast for n ≤ ~2000 |
| 3 | β | Conjugate normal | Dense Cholesky |
| 4 | σ² | Conjugate inverse-Gamma | Direct draw |
| 5 | ρ | **Slice sampling** (Neal stepping-out) | Collapsed conditional, Lanczos log\|P\| |
| 6 | α | **Slice sampling** (JAX-compiled) | On log(α) scale |

### Key features

- **Lanczos log|P|**: Stochastic log-determinant estimation for the collapsed ρ conditional. A fixed random key is used within each slice step to ensure deterministic density evaluations during stepping-out and shrinkage.
- **No MALA tuning needed**: Slice sampling is self-tuning — no step-size adaptation required.
- **Full JIT compilation**: All 6 blocks are composed into a single `@eqx.filter_jit` function, paying the Python→JAX dispatch cost only once per iteration.

```python
# JAX Gibbs with slice sampling (default)
model_jax = SARNegBinLatent(y=y, X=X, W=W)
idata_jax = model_jax.fit(
    gibbs_method="jax_dense",
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
    pg_n_terms=10,       # Pólya–Gamma approximation order
    n_probes=5,           # Lanczos probe vectors for log|P|
    lanczos_deg=15,       # Lanczos iteration depth
)

az.summary(idata_jax, var_names=["rho", "beta", "sigma", "alpha"])
```

## Log-Determinant Method Choices

The spatial Jacobian $\log|I - \rho W|$ is needed for the ρ slice sampler in `SARNegBinLatent`. `bayespecon` supports several approximation methods:

| Method | `logdet_method` | Precompute | Per-ρ cost | Best for |
|---|---|---|---|---|
| Eigenvalue | `"eigenvalue"` | O(n³) | O(n) | n < 500 |
| Chebyshev | `"chebyshev"` | O(n³) or O(nm) | O(m) | default for n ≥ 500 |

The logdet method is set at model construction time and is shared by all samplers.

## InferenceData Compatibility and ArviZ Diagnostics

Both `SARNegativeBinomial` and `SARNegBinLatent` produce the same `az.InferenceData` output with `posterior`, `log_likelihood`, and `observed_data` groups. All ArviZ diagnostics work seamlessly.

In [ ]:
# ArviZ diagnostics work with both NUTS and Gibbs-produced InferenceData
print("Groups:", idata_gibbs.groups())
print()

# LOO cross-validation
loo = az.loo(idata_gibbs)
print(f"LOO: elpd = {loo.elpd_loo:.2f}, SE = {loo.se:.2f}")
print()

# WAIC
waic = az.waic(idata_gibbs)
print(f"WAIC: elpd = {waic.elpd_waic:.2f}, SE = {waic.se:.2f}")
print()

# Summary
az.summary(idata_gibbs, var_names=["rho", "sigma", "alpha"])

In [ ]:
az.plot_trace(idata_gibbs, var_names=["rho", "sigma", "alpha"])

## Overdispersion Parameter Diagnostics

The NB2 dispersion parameter $\alpha$ controls the variance-mean relationship:

$$\mathrm{Var}(y_i) = \mu_i + \mu_i^2 / \alpha$$

- **$\alpha \to \infty$**: Poisson limit (no overdispersion)
- **Small $\alpha$**: Strong overdispersion

Monitoring the posterior of $\alpha$ helps assess whether the NB model is necessary or a Poisson model would suffice.

In [ ]:
# Posterior summary for alpha
alpha_samples = idata_gibbs.posterior["alpha"].values.flatten()
print(
    f"alpha posterior: mean = {alpha_samples.mean():.3f}, "
    f"median = {np.median(alpha_samples):.3f}, "
    f"95% CI = [{np.percentile(alpha_samples, 2.5):.3f}, {np.percentile(alpha_samples, 97.5):.3f}]"
)
print("True alpha = 2.0")
print()

# Compare observed variance-mean ratio with what NB2 predicts
mu_hat = np.exp(
    np.clip(
        np.mean(idata_gibbs.posterior["eta_norm"].values, axis=(0, 1))
        if "eta_norm" in idata_gibbs.posterior
        else X @ idata_gibbs.posterior["beta"].mean(dim=["chain", "draw"]).values,
        -30,
        30,
    )
)
print(f"Observed: mean(y) = {y.mean():.1f}, var(y) = {y.var():.1f}")
print(f"Variance-mean ratio: {y.var() / y.mean():.2f}")
print(
    f"NB2 predicted ratio at posterior mean alpha: 1 + mean(y)/alpha = {1 + y.mean() / alpha_samples.mean():.2f}"
)

## Summary

The negative binomial and spatial logit model estimation in `bayespecon` provides:

### NegBin Models

1. **Two model classes**: `SARNegativeBinomial` (NUTS, non-centred parameterisation) and `SARNegBinLatent` (Pólya–Gamma Gibbs, structural form)
2. **Same output format**: Both produce `az.InferenceData` with `posterior`, `log_likelihood`, `observed_data`
3. **Multiple Gibbs backends**: CHOLMOD (fastest), SPLU (fallback), JAX dense (full JIT)
4. **Full ArviZ compatibility**: LOO, WAIC, summary, plot
5. **Full diagnostics compatibility**: Bayesian LM tests, spatial diagnostics decision tree
6. **Native DGP**: `simulate_sar_negbin` generates data matching either model specification

### Spatial Logit Models

1. **Two model classes**: `SARSpatialLogit` (spatial lag) and `SEMSpatialLogit` (spatial error)
2. **PG-Gibbs only**: No NUTS path — PG augmentation makes all blocks conjugate or collapsed
3. **Same output format**: `az.InferenceData` with `posterior`, `log_likelihood`, `observed_data`
4. **Multiple Gibbs backends**: CHOLMOD, SPLU, JAX dense (same as NegBin)
5. **Fitted probabilities**: `fitted_probabilities()` returns P(y=1) at posterior mean
6. **Native DGP**: `simulate_sar_logit` and `simulate_sem_logit`

```python
# Quick reference — NegBin
from bayespecon.dgp import simulate_sar_negbin
from bayespecon.models import SARNegBinLatent, SARNegativeBinomial

data = simulate_sar_negbin(n=10, rho=0.5, alpha=2.0, sigma2=0.5, seed=42)

# NUTS (small n, need PyMC model graph)
model = SARNegativeBinomial(y=data["y"], X=data["X"], W=data["W_graph"])
idata = model.fit(draws=2000, tune=1000, chains=4, target_accept=0.9)

# PG-Gibbs (large n, need speed)
model = SARNegBinLatent(y=data["y"], X=data["X"], W=data["W_graph"])
idata = model.fit(draws=2000, tune=1000, chains=4, n_jobs=-1)

# Quick reference — Spatial Logit
from bayespecon.dgp import simulate_sar_logit, simulate_sem_logit
from bayespecon.models import SARSpatialLogit, SEMSpatialLogit

# SAR-logit (spatial lag on latent log-odds)
data = simulate_sar_logit(n=10, rho=0.5, beta=np.array([0.3, 1.0]), seed=42)
model = SARSpatialLogit(y=data["y"], X=data["X"], W=data["W_graph"])
idata = model.fit(draws=2000, tune=1000, chains=4, n_jobs=-1)

# SEM-logit (spatial error in latent log-odds)
data = simulate_sem_logit(n=10, lam=0.5, beta=np.array([0.3, 1.0]), seed=42)
model = SEMSpatialLogit(y=data["y"], X=data["X"], W=data["W_graph"])
idata = model.fit(draws=2000, tune=1000, chains=4, n_jobs=-1)

# JAX backend (full JIT compilation)
idata = model.fit(draws=2000, tune=1000, chains=4, gibbs_method="jax_dense")

# Fitted probabilities
probs = model.fitted_probabilities()

# Spatial diagnostics
model.spatial_diagnostics()
model.spatial_diagnostics_decision()
```